# 03 - Construction de la GOLD DATA

Objectif : produire une table analytique unique a granularite pays-destination, utilisable par la BI, le scoring metier et le dashboard.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
from src.feature_engineering import (
    aggregate_campaigns, aggregate_external, aggregate_reviews,
    build_country_market_features, build_gold_data
)

processed = ROOT / "data" / "processed"
destinations = pd.read_csv(processed / "destinations_clean.csv")
market = pd.read_csv(processed / "market_clean.csv", parse_dates=["month"])
reviews = pd.read_csv(processed / "reviews_clean.csv")
external = pd.read_csv(processed / "external_clean.csv")
campaigns = pd.read_csv(processed / "campaigns_clean.csv")

## 1. Aggregations par granularite

In [ ]:
review_agg = aggregate_reviews(reviews)
external_agg = aggregate_external(external)
campaign_agg = aggregate_campaigns(campaigns)
market_features = build_country_market_features(market)

display(review_agg.head())
display(external_agg.head())
display(campaign_agg.head())
display(market_features.head())

Decision : les reviews, facteurs externes et campagnes sont agreges avant jointure. Cela evite de multiplier les lignes destination par le nombre d'avis ou de signaux externes.

## 2. Variables derivees et valeur metier

In [ ]:
variable_rationale = pd.DataFrame([
    ["demand_growth", "Mesure la dynamique recente du marche pays", "Permet de pousser les pays en acceleration"],
    ["tourism_score", "Combine attractivite, visiteurs, demande et volume d'avis", "Mesure le potentiel global"],
    ["quality_score", "Combine rating source et score reviews", "Evite de recommander une destination mal percue"],
    ["sentiment_score", "Convertit le sentiment social en signal numerique", "Capte la tonalite reputationale"],
    ["attractiveness_cost_ratio", "Attractivite par unite de cout", "Integre l'efficience budgetaire"],
    ["weather_penalty", "Risque meteo externe", "Evite les destinations defavorables"],
    ["campaign_efficiency", "ROI rapporte au budget connu", "Valorise les apprentissages marketing"],
    ["marketing_priority", "Score final pondere", "Classement actionnable pour le comite marketing"],
], columns=["Variable", "Definition", "Interet business"])
variable_rationale

## 3. Generation GOLD DATA

In [ ]:
gold = build_gold_data(destinations, market, reviews, external, campaigns)
display(gold.head())
display(gold.describe().T)

In [ ]:
selected = ["forecasted_demand", "quality_score", "sentiment_score", "weather_penalty", "campaign_efficiency", "marketing_priority"]
plt.figure(figsize=(9, 6))
sns.heatmap(gold[selected].corr(), annot=True, cmap="vlag", center=0)
plt.title("Correlation des variables de scoring")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=gold,
    x="quality_score",
    y="forecasted_demand",
    hue="country",
    size="marketing_priority",
    sizes=(30, 250),
)
plt.title("Qualite vs demande prevue, taille = priorite marketing")
plt.tight_layout()

Interpretation : une destination prioritaire n'est pas seulement une destination populaire. Elle doit combiner demande, qualite, sentiment, efficience cout/campagne et risque meteo acceptable.

## 4. Sauvegarde et controles finaux

In [ ]:
gold_path = ROOT / "data" / "gold" / "gold_tourism_data.csv"
gold_path.parent.mkdir(parents=True, exist_ok=True)
gold.to_csv(gold_path, index=False)

pd.DataFrame({
    "rows": [len(gold)],
    "columns": [gold.shape[1]],
    "missing_cells": [gold.isna().sum().sum()],
    "unique_countries": [gold["country"].nunique()],
    "unique_destinations": [gold["destination"].nunique()],
})